In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

import torch

sys.path.append(".")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


In [ ]:
import glob
from pathlib import Path

from huggingface_hub import snapshot_download

from config import load_run_config
from finetuning import aggregate as agg
from finetuning.experiments import (
    run_experiment, DatasetConfig, ExperimentConfig,
    ALLSIDES_BASE_MEDIA_SPLIT,ALLSIDES_BASE_RANDOM_SPLIT
)
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
def find_tlp_checkpoints(config_glob="run_configs/tlp_*.yaml"):
    """Locate the checkpoint each tlp_* pretraining run left behind.
    """
    checkpoints = []
    for config_path in sorted(glob.glob(config_glob)):
        label = Path(config_path).stem
        output_dir = Path(load_run_config(config_path).output_dir)
        epochs = sorted(
            output_dir.glob("epoch-*.pt"),
            key=lambda p: int(p.stem.split("-")[1]),
        )
        if not epochs:
            print(f"  SKIP {label}: no epoch-*.pt under {output_dir}")
            continue
        print(f"  {label}: {epochs[-1]}")
        checkpoints.append((str(epochs[-1]), label))
    return checkpoints


print("tlp checkpoints:")
TLP_CHECKPOINTS = find_tlp_checkpoints()
print(f"\nfound {len(TLP_CHECKPOINTS)} of 4")


In [ ]:
BASELINES = [
    (BERT, "bert"),
    (BART, "bart"),
    (ROBERTA, "roberta"),
    (POLITICS, "politics"),
    (ideology_pt, "ideology"),
]


def run_all_models(models, dataset, output_prefix, seed):
    """Fine-tune each of `models` on one bias dataset, under one seed.
    """
    os.makedirs(output_prefix, exist_ok=True)
    results = {}
    for model_ref, model_name in models:
        cfg = DatasetConfig(custom_dataset=dataset)
        exp = ExperimentConfig(patience=3, num_epochs=15, save_model=False, seed=seed)
        print(f"\n{'='*60}")
        print(f"Model: {model_name}  |  Dataset: {dataset} |  Seed: {seed}")
        print('='*60)
        results[model_name] = run_experiment(
            model=model_ref,
            loc=output_prefix,
            dataset_config=cfg,
            experiment_config=exp,
            model_name=model_name,
        )
    return results


In [ ]:
seeds = [42, 1, 13, 1234, 6789]

SPLITS = {
    "media_split": ALLSIDES_BASE_MEDIA_SPLIT,
    "random_split": ALLSIDES_BASE_RANDOM_SPLIT,
}
SPLIT = "media_split"

DATASET = SPLITS[SPLIT]
RESULTS_ROOT = f"results_undersampling/{SPLIT}"
print(f"{SPLIT}: {DATASET} -> {RESULTS_ROOT}")


In [ ]:
for seed in seeds:
    print(f"\n{'#'*60}")
    print(f"Baselines, {SPLIT}, seed: {seed}")
    print('#'*60)
    run_all_models(
        models=BASELINES,
        dataset=DATASET,
        output_prefix=f"{RESULTS_ROOT}/seed_{seed}",
        seed=seed,
    )


In [ ]:
for seed in seeds:
    print(f"\n{'#'*60}")
    print(f"tlp checkpoints, {SPLIT}, seed: {seed}")
    print('#'*60)
    run_all_models(
        models=TLP_CHECKPOINTS,
        dataset=DATASET,
        output_prefix=f"{RESULTS_ROOT}/seed_{seed}",
        seed=seed,
    )


## Cross-seed analysis

Everything below reads the per-seed metrics JSONs off disk — no model, no GPU. The
runs above do not need to have happened in this session.


In [ ]:
results = agg.discover_results(RESULTS_ROOT)
print(f"{len(results)} runs under {RESULTS_ROOT}")

# Check this before reading anything below: a model missing seeds gets a mean
# over fewer runs, and n is the only other place that shows up.
agg.coverage(results)


In [ ]:
# Headline table: mean (std) across seeds, per model. Percentages, as written.
agg.summary_table(results)


In [ ]:
# Per model: mean (std), the majority-vote ensemble over seeds, and how much the
# seeds disagree per example.
agg.print_reports(results)


In [ ]:
# Ensemble gain and stability side by side. A model whose majority vote beats its
# mean seed by a lot is one whose errors are mostly seed noise. polarity_flips is
# the share of those disagreements that are 0<->2 rather than off-by-one.
import pandas as pd

rows = []
for model, runs in sorted(agg.by_model(results).items()):
    vote = agg.majority_vote(runs)
    spread = agg.disagreement(runs) if len(runs) > 1 else {}
    errors = agg.error_breakdown(runs)
    flips = errors[errors.distance == agg.POLARITY_FLIP_DISTANCE]
    rows.append({
        "model": model,
        "n_seeds": len(runs),
        "mean_seed_f1": vote["mean_single_seed_f1_macro"],
        "vote_f1": vote["f1_macro"],
        "gain": vote["gain_f1_macro"],
        "unanimity": spread.get("unanimity_rate"),
        "pairwise_disagree": spread.get("mean_pairwise_disagreement"),
        "polarity_flips": spread.get("polarity_flip_share"),
        "fleiss_kappa": spread.get("fleiss_kappa"),
        "flip_share_of_errors": round(float(flips["share_of_errors"].sum()), 2),
    })
pd.DataFrame(rows).sort_values("vote_f1", ascending=False)


In [ ]:
# Which classes get confused, per model. Labels are ordinal (0=left, 1=center,
# 2=right), so 0<->2 is a polarity flip and the rest are off-by-one into or out
# of center. Two models with the same F1 can fail in very different ways here.
MODEL = "tlp_theme_tone_16"

runs = agg.by_model(results).get(MODEL)
if not runs:
    print(f"no runs for {MODEL}; available: {sorted(agg.by_model(results))}")
else:
    print(f"{MODEL}: errors by confusion (mean per seed, and the ensemble)")
    display(agg.error_breakdown(runs))
    print("seed-vs-seed disagreement by class pair")
    display(agg.disagreement_breakdown(runs))
    print("confusion matrix, mean over seeds (rows = true, cols = predicted)")
    display(agg.confusion_frame(runs, source="mean").round(1))


In [ ]:
# Error analysis: the examples one model's seeds most disagree on. These are where
# seed choice alone decides the answer, so they are the useful ones to read.
MODEL = "tlp_theme_tone_16"

runs = agg.by_model(results).get(MODEL)
if not runs:
    print(f"no runs for {MODEL}; available: {sorted(agg.by_model(results))}")
else:
    frame = agg.per_example_frame(runs)
    print(f"{MODEL}: {(frame.n_distinct > 1).sum()} of {len(frame)} examples contested")
    display(frame.sort_values(["n_distinct", "n_correct"], ascending=[False, True]).head(20))
